<a href="https://colab.research.google.com/github/7ING-IC/Practice_LLM/blob/main/prac_LangChain_model%26prompt%26parsers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai

In [ ]:
pip install -U langchain langchain-openai langchain-community

In [ ]:
pip install -U langchain langchain-core langchain-openai

In [ ]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv

# 加载环境变量
_ = load_dotenv(find_dotenv())

# 或手动设置 key
os.environ["OPENAI_API_KEY"] = "Your api key"

# 初始化客户端
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 示例请求
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What is 1+1?"}],
)

print(response.choices[0].message.content)


1 + 1 equals 2.


In [ ]:
def get_completion(prompt, model="gpt-3.5-turbo"):
  client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
  message = [{"role": "user", "content": prompt}]
  response = client.chat.completions.create(
    model=model,
    messages=message,
    temperature=0,
  )
  return response.choices[0].message.content

In [ ]:
 get_completion("what is 1+1?")

'1+1 equals 2.'

In [ ]:
from langchain_openai import ChatOpenAI

In [ ]:
chat = ChatOpenAI(temperature=0.0)
chat

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x7f5c0e7a64e0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7f5c0e7a65d0>, root_client=<openai.OpenAI object at 0x7f5c0e7a5130>, root_async_client=<openai.AsyncOpenAI object at 0x7f5c0e7a6540>, temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

In [ ]:
template_string= """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text:```{text}```
"""

In [ ]:
# prompts + chat models 最新结构
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt_template = ChatPromptTemplate.from_template(template_string)

In [ ]:
prompt_template.messages[0].prompt

PromptTemplate(input_variables=['style', 'text'], input_types={}, partial_variables={}, template='Translate the text that is delimited by triple backticks into a style that is {style}. text:```{text}```\n')

In [ ]:
prompt_template.messages[0].prompt.input_variables

['style', 'text']

In [ ]:
customer_style = """Chinese in a calm and respectful tone"""
customer_email = """Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! \
And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. \
I need yer help right now, matey!"""
# 多行字符串（可跨行、可包含换行符）

In [ ]:
customer_messages = prompt_template.format_messages(style=customer_style, text=customer_email)

In [ ]:
print(type(customer_messages))
print(type(customer_messages[0]))

<class 'list'>
<class 'langchain_core.messages.human.HumanMessage'>


In [ ]:
customer_response = chat.invoke(customer_messages)
print(customer_response.content)

尊敬的朋友，我感到非常生气，因为我的搅拌机盖飞出去，把厨房墙壁都溅满了果汁！更糟糕的是，保修不包括清理厨房的费用。望您能立即帮助我，伙计！感激不尽。


In [ ]:
{
    "gift": False,
  "delivery_days": 5,
  "price_value": "pretty affordable!"
}

{'gift': False, 'delivery_days': 5, 'price_value': 'pretty affordable!'}

In [ ]:
{'gift': False, 'deluvery_days': 5, 'price_value': 'pretty affordable!'}

{'gift': False, 'deluvery_days': 5, 'price_value': 'pretty affordable!'}

In [ ]:
customer_review = """This leaf blower is pretty amazing. It has four settings: candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been using it every other morning to clear the leaves on our lawn.\
It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features.
"""
review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? Answer True or False.
delivery_days: How many days did it take for the product to arrive?
price_value: Extract any sentences about the value or price, and what the reviewer thinks of it.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""
# 评论模版要求LLM将客户评论作为输入，提取这三个字段，然后将输出格式化为JSON，带有一下键

In [ ]:
# 将上述内容封装在langChain
prompt_template = ChatPromptTemplate.from_template(review_template)
prompt_template

ChatPromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='For the following text, extract the following information:\n\ngift: Was the item purchased as a gift for someone else? Answer True or False.\ndelivery_days: How many days did it take for the product to arrive?\nprice_value: Extract any sentences about the value or price, and what the reviewer thinks of it.\n\nFormat the output as JSON with the following keys:\ngift\ndelivery_days\nprice_value\n\ntext: {text}\n'), additional_kwargs={})])

In [ ]:
messages = prompt_template.format_messages(text=customer_review)
chat = ChatOpenAI(temperature=0.0)
response = chat.invoke(messages)

print(response.content)

{
    "gift": true,
    "delivery_days": 2,
    "price_value": "It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."
}


In [ ]:
type(response.content)
# 只是看起来像JSON，但不是dictionary类型
# JSON 是一种简单、通用、高效的数据“普通话”。
# 它让不同的系统（比如用 Python 写的后端和用 JavaScript 写的前端）
# 能够用一种彼此都懂的方式“交谈”。

str

In [ ]:
#  Parser（解析器） 就是一个把模型输出的自然语言文本转化为结构化数据的组件。

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

In [ ]:
# 1️⃣ 定义数据结构（替代 ResponseSchema）
class GiftInfo(BaseModel):
    gift: str = Field(description="礼物是否合适，例如 Yes/No 或一个短语")
    delivery_days: int = Field(description="送达所需的天数")
    price_value: str = Field(description="价格水平，例如 'expensive' 或 'cheap'")

# 2️⃣ 创建解析器
parser = PydanticOutputParser(pydantic_object=GiftInfo)
format_instructions = parser.get_format_instructions()

In [ ]:
print(format_instructions)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"disease": {"description": "疾病名称", "title": "Disease", "type": "string"}, "confidence": {"description": "置信度 0~1", "title": "Confidence", "type": "number"}}, "required": ["disease", "confidence"]}
```


In [ ]:
review_template_2=review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? Answer True or False.
delivery_days: How many days did it take for the product to arrive?
price_value: Extract any sentences about the value or price, and what the reviewer thinks of it.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
{format_instructions}
"""

# 定义提示词
prompt = ChatPromptTemplate.from_template(template = review_template_2)
messages = prompt.format_messages(text=customer_review, format_instructions=format_instructions)

In [ ]:
print(messages[0].content)

For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? Answer True or False.
delivery_days: How many days did it take for the product to arrive?
price_value: Extract any sentences about the value or price, and what the reviewer thinks of it.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: This leaf blower is pretty amazing. It has four settings: candle blower, gentle breeze, windy city, and tornado. It arrived in two days, just in time for my wife's anniversary present. I think my wife liked it so much she was speechless. So far I've been the only one using it, and I've been using it every other morning to clear the leaves on our lawn.It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties"

In [ ]:
response= chat.invoke(messages)

In [ ]:
print(response.content)

{
  "gift": "True",
  "delivery_days": 2,
  "price_value": "It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."
}


In [ ]:
output_dict = parser.parse(response.content)
# 新版schema是先创建了了一个类（GiftInfo），基于pydantic模型
# 因此此时outpu_dict是实例对象
# 利用pydantic里的model_dump()方法将其转为dict类型
output_dict = output_dict.model_dump()
output_dict

{'gift': 'True',
 'delivery_days': 2,
 'price_value': "It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."}

In [ ]:
type(output_dict)

dict

In [ ]:
output_dict.get('delivery_days')

2